# EDA de calidad del aire en Concón, Quintero y Puchuncaví

**Avance 2 · Periodo 2023–2025 · Descarga SINCA del 4 de octubre de 2026**

Este notebook explora mediciones reales de cuatro estaciones históricas. Todas las mediciones disponibles de contaminantes de esta versión figuran como **no validadas**. Los resultados son preliminares y descriptivos. No se clasifican incumplimientos normativos ni se atribuyen emisiones a empresas.

Se mantiene el problema del Avance 1 y se ajusta el alcance: la estación histórica Concón sustituye explícitamente a Concón Oeste. Centro Quintero es el nombre oficial en SINCA. Temperatura y humedad se analizan solo en Concón.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown

# Admite ejecución desde la raíz o desde notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'data/processed').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/processed').exists(), 'Ejecuta desde dataset_cqp o notebooks'
sys.path.insert(0, str(ROOT / 'src'))
import eda
pio.renderers.default = 'plotly_mimetype'
pd.set_option('display.max_columns', 20)
df = eda.load_data()
print(eda.QUESTION)
print(f'{len(df):,} filas; {df.shape[1]} columnas; {df.fecha_hora.min()} a {df.fecha_hora.max()}')

¿Cómo varían las concentraciones horarias de MP2.5, MP10 y SO₂ entre Ventanas, La Greda, Centro Quintero y Concón, según hora del día, época del año y condiciones meteorológicas disponibles, durante 2023–2025?
105,216 filas; 32 columnas; 2023-01-01 00:00:00 a 2025-12-31 23:00:00


## 1. Estructura y preparación

Y es la concentración horaria de cada contaminante. X incluye estación, hora y meteorología disponible; T es el periodo 2023–2025. MP2.5 se conserva en µg/m³, MP10 en µg/m³N y SO₂ en ppb. No se mezclan sus escalas.

Los TXT originales y sus hashes están en `data/raw` y `docs/download_manifest.json`. La preparación no imputa valores, no sustituye faltantes por cero y no elimina extremos dentro de rango. Los timestamps mantienen la etiqueta de SINCA sin asumir un offset UTC. La grilla de horas hace visibles los faltantes; una fila no significa que todas sus mediciones estén presentes.

In [2]:
display(df.dtypes.rename('tipo').to_frame())
assert not df.duplicated(['estacion_id', 'fecha_hora']).any()
display(df.groupby(['estacion', 'comuna']).size().rename('posiciones_horarias').reset_index())
quality = pd.read_csv(ROOT / 'docs/calidad_por_serie.csv')
display(quality[['estacion','variable','horas_esperadas','horas_con_dato','cobertura_pct','duplicados','fuera_rango']])

,tipo
fecha_hora,datetime64[us]
estacion_id,int64
codigo_serie,int64
estacion,str
comuna,str
direccion_viento_grados,float64
humedad_relativa_pct,float64
mp10_ug_m3n,float64
mp25_ug_m3,float64
so2_ppb,float64


,estacion,comuna,posiciones_horarias
0,Centro Quintero,Quintero,26304
1,Concón,Concón,26304
2,La Greda,Puchuncaví,26304
3,Ventanas,Puchuncaví,26304


,estacion,variable,horas_esperadas,horas_con_dato,cobertura_pct,duplicados,fuera_rango
0,La Greda,so2_ppb,26304,25981,98.77,0,0
1,La Greda,mp10_ug_m3n,26304,26173,99.50,0,0
2,La Greda,mp25_ug_m3,26304,26101,99.23,0,0
3,La Greda,direccion_viento_grados,26304,26220,99.68,0,0
4,La Greda,velocidad_viento_m_s,26304,26216,99.67,0,0
5,Concón,so2_ppb,26304,26005,98.86,0,0
6,Concón,mp10_ug_m3n,26304,25880,98.39,0,0
7,Concón,mp25_ug_m3,26304,25309,96.22,0,0
8,Concón,humedad_relativa_pct,26304,26234,99.73,0,0
9,Concón,temperatura_c,26304,26234,99.73,0,0


## 2. Cobertura y clasificación

**Pregunta:** ¿qué estaciones y meses tienen menos información?

La cobertura se calcula como mediciones disponibles / horas esperadas. La pérdida puede depender del periodo o de la operación del equipo. No suponemos que sea aleatoria.

In [3]:
display(df.groupby(['estacion', 'mp25_ug_m3_estado']).size().rename('horas').reset_index())
eda.coverage_chart(df, 'mp25_ug_m3').show()
coverage = eda.monthly_coverage(df, 'mp25_ug_m3')
display(coverage.sort_values('cobertura_pct').head(10))

,estacion,mp25_ug_m3_estado,horas
0,Centro Quintero,no_validado,26248
1,Centro Quintero,sin_dato,56
2,Concón,no_validado,25309
3,Concón,sin_dato,995
4,La Greda,no_validado,26101
5,La Greda,sin_dato,203
6,Ventanas,no_validado,23635
7,Ventanas,sin_dato,2669


,estacion,periodo,n,esperadas,cobertura_pct
142,Ventanas,2025-11,521,720,72.361111
132,Ventanas,2025-01,578,744,77.688172
130,Ventanas,2024-11,577,720,80.138889
140,Ventanas,2025-09,585,720,81.250000
131,Ventanas,2024-12,608,744,81.720430
115,Ventanas,2023-08,609,744,81.854839
141,Ventanas,2025-10,617,744,82.930108
139,Ventanas,2025-08,634,744,85.215054
135,Ventanas,2025-04,618,720,85.833333
134,Ventanas,2025-03,640,744,86.021505


**Observación:** MP2.5 de Ventanas tiene 89,85 % de cobertura global y el mes con menor cobertura es noviembre de 2025, con 72,36 %. Los otros contaminantes y estaciones tienen coberturas distintas.

**Interpretación:** las diferencias de disponibilidad pueden alterar comparaciones temporales o espaciales. Para comparar estaciones, el análisis siguiente usa únicamente timestamps donde todas tienen medición del contaminante. Esto controla la población temporal comparada, aunque no elimina un posible sesgo del subconjunto.

## 3. Estadísticas descriptivas y horas comunes

**Pregunta:** ¿qué niveles, dispersión y valores extremos presenta cada contaminante?

Se usa mediana junto a media y percentil 95 para distinguir el centro de las colas. Las horas comunes se calculan por contaminante, por lo que la población temporal de MP2.5 no necesariamente coincide con la de MP10 o SO₂.

In [4]:
common = {}
for variable in eda.VARIABLES:
    common[variable] = eda.common_hours(df, variable)
    print('\n', eda.axis_label(variable), 'Horas comunes:', common[variable].fecha_hora.nunique())
    display(eda.station_summary(common[variable], variable).round(3))


 MP2.5 (µg/m³) Horas comunes: 22671


,estacion,horas_esperadas,n,media,mediana,desviacion,minimo,maximo,p95,cobertura_pct
0,Centro Quintero,22671,22671,15.895,13.0,12.687,1.0,157.0,41.0,100.0
1,Concón,22671,22671,13.855,12.0,8.094,0.0,126.0,29.0,100.0
2,La Greda,22671,22671,15.874,13.0,10.886,1.0,118.0,37.0,100.0
3,Ventanas,22671,22671,18.221,16.0,12.192,0.0,192.0,41.0,100.0



 MP10 (µg/m³N) Horas comunes: 25223


,estacion,horas_esperadas,n,media,mediana,desviacion,minimo,maximo,p95,cobertura_pct
0,Centro Quintero,25223,25223,38.050,35.0,18.457,1.0,223.0,71.0,100.0
1,Concón,25223,25223,31.871,29.0,19.278,0.0,261.0,67.0,100.0
2,La Greda,25223,25223,39.024,34.0,27.352,0.0,554.0,86.0,100.0
3,Ventanas,25223,25223,43.603,40.0,25.842,0.0,396.0,87.0,100.0



 SO₂ (ppb) Horas comunes: 24993


,estacion,horas_esperadas,n,media,mediana,desviacion,minimo,maximo,p95,cobertura_pct
0,Centro Quintero,24993,24993,3.566,2.92,3.944,0.10,222.05,7.660,100.0
1,Concón,24993,24993,4.544,3.62,4.363,0.53,191.94,10.694,100.0
2,La Greda,24993,24993,2.914,2.77,1.514,0.00,90.04,4.830,100.0
3,Ventanas,24993,24993,2.913,2.71,1.622,0.41,87.58,4.250,100.0


## 4. Distribución y diferencias entre estaciones

**Pregunta:** ¿las estaciones tienen distribuciones similares de MP2.5?

El histograma usa porcentajes dentro de cada estación y bins compartidos. El boxplot usa todas las observaciones de las horas comunes, aunque no dibuja puntos extremos individualmente.

In [5]:
mp25 = common['mp25_ug_m3']
eda.histogram_chart(mp25, 'mp25_ug_m3').show()
eda.box_chart(mp25, 'mp25_ug_m3').show()

**Observación:** en 22.671 horas comunes, las medianas de MP2.5 son 16 µg/m³ en Ventanas, 13 en Centro Quintero y La Greda, y 12 en Concón. Las medias son mayores que las medianas en las cuatro estaciones.

**Interpretación:** existen colas hacia concentraciones altas y diferencias en el centro de la distribución. Ventanas muestra una mediana mayor en los puntos y horas comparados. No se deduce de ello que toda Puchuncaví tenga esa exposición ni se clasifica su calidad del aire mediante una norma.

In [6]:
for variable in ['mp10_ug_m3n','so2_ppb']:
    eda.box_chart(common[variable], variable).show()

**Observación:** en horas comunes, Ventanas tiene la mayor mediana de MP10 (40 µg/m³N) y Concón la mayor mediana de SO₂ (3,62 ppb).

**Interpretación:** el orden espacial cambia según el contaminante. El proyecto debe mantener análisis separados y evitar un ranking único de “comuna más contaminada”.

## 5. Evolución temporal y comparación anual

**Pregunta:** ¿cómo cambian las concentraciones durante el periodo?

La media diaria se muestra solo si el día tiene al menos 18 horas observadas. Es un criterio exploratorio sobre 24 posiciones esperadas, no un cálculo normativo. Se mantienen huecos cuando no hay cobertura suficiente.

In [7]:
eda.daily_chart(df, 'mp25_ug_m3').show()
annual = mp25.groupby(['estacion','anio'])['mp25_ug_m3'].agg(n='count', mediana='median', media='mean').reset_index()
display(annual.round(3))

,estacion,anio,n,mediana,media
0,Centro Quintero,2023,7889,15.0,17.844
1,Centro Quintero,2024,7708,13.0,16.249
2,Centro Quintero,2025,7074,9.0,13.336
3,Concón,2023,7889,12.0,12.390
4,Concón,2024,7708,11.0,12.108
5,Concón,2025,7074,15.0,17.394
6,La Greda,2023,7889,13.0,15.145
7,La Greda,2024,7708,13.0,15.912
8,La Greda,2025,7074,13.0,16.646
9,Ventanas,2023,7889,16.0,18.204


**Interpretación:** las series permiten ubicar aumentos y periodos incompletos. Las estadísticas anuales de horas comunes permiten investigar cambios, pero los conjuntos de horas pueden tener composición distinta entre años. No se atribuye ningún cambio a la inauguración de la red pública: estas son estaciones históricas diferentes y no existe un diseño causal.

## 6. Patrones por hora y época del año

**Pregunta:** ¿en qué horarios aparecen mayores concentraciones típicas?

El perfil usa medianas por hora y las mismas horas entre estaciones. Se muestran grupos con al menos 24 observaciones. El mapa mes/hora usa una estación y agrupa los años seleccionados; se ocultan celdas con menos de 12 observaciones.

In [8]:
eda.hourly_chart(mp25,'mp25_ug_m3').show()
profile = eda.hourly_profile(mp25,'mp25_ug_m3')
display(profile[profile.estacion.eq('Ventanas')])
eda.month_hour_chart(df,'mp25_ug_m3','Ventanas').show()

,estacion,hora,mediana,n
72,Ventanas,0,17.0,907
73,Ventanas,1,16.0,940
74,Ventanas,2,14.5,962
75,Ventanas,3,13.0,970
76,Ventanas,4,12.0,972
77,Ventanas,5,11.0,957
78,Ventanas,6,11.0,941
79,Ventanas,7,10.0,916
80,Ventanas,8,10.0,851
81,Ventanas,9,11.0,838


**Observación:** Ventanas presenta un máximo mediano de MP2.5 de 24 µg/m³ a las 14:00 y un mínimo de 10 µg/m³ a las 07:00 y 08:00. Centro Quintero presenta medianas mayores de noche que durante la tarde en el perfil agregado.

**Interpretación:** el horario importa y el patrón no es igual en todas las estaciones. El mapa por mes permite investigar si el perfil cambia durante el año. Debemos comprobarlo por año antes de afirmar estabilidad estacional o identificar mecanismos.

## 7. Concentración y condiciones meteorológicas

**Pregunta:** ¿la velocidad del viento tiene una asociación consistente con MP2.5?

Se calculan correlaciones de Spearman por estación sobre todos los pares disponibles, usando rangos medios. El scatter representa una muestra determinística de hasta 1.500 pares por estación para mantener legibilidad. El muestreo afecta solo el dibujo. No se hacen pruebas de significancia: las observaciones horarias están autocorrelacionadas.

In [9]:
eda.scatter_chart(df,'mp25_ug_m3').show()
display(eda.relationships(df,'mp25_ug_m3').round(3))
for predictor in ['temperatura_c','humedad_relativa_pct']:
    print(eda.METEOROLOGY[predictor][0])
    display(eda.relationships(df,'mp25_ug_m3',predictor).round(3))

,estacion,n_pares,spearman
0,Centro Quintero,26107,-0.214
1,Concón,25295,0.096
2,La Greda,26091,-0.209
3,Ventanas,23477,0.235


Temperatura, solo Concón


,estacion,n_pares,spearman
0,Centro Quintero,0,NaN
1,Concón,25309,0.019
2,La Greda,0,NaN
3,Ventanas,0,NaN


Humedad relativa, solo Concón


,estacion,n_pares,spearman
0,Centro Quintero,0,NaN
1,Concón,25309,-0.102
2,La Greda,0,NaN
3,Ventanas,0,NaN


**Observación:** la asociación MP2.5–velocidad del viento es positiva en Ventanas (ρ≈0,235) y ligeramente positiva en Concón (ρ≈0,096), y negativa en La Greda (ρ≈−0,209) y Centro Quintero (ρ≈−0,214).

**Interpretación:** los datos no sostienen una relación simple y uniforme entre mayor viento y menor MP2.5. Dirección, horario y época son posibles dimensiones a investigar. Esto es asociación, sin atribución causal. Temperatura y humedad solo permiten analizar Concón.

## 8. Anomalías y límites

**Pregunta:** ¿qué registros necesitan revisión adicional?

Se listan máximos por contaminante con su timestamp y estado. Un valor elevado dentro de rango puede ser un evento real o una anomalía instrumental. Sin validación, no se elige entre esas explicaciones.

In [10]:
for variable in eda.VARIABLES:
    print(eda.axis_label(variable))
    display(df.nlargest(5,variable)[['fecha_hora','estacion',variable,variable+'_estado']])

MP2.5 (µg/m³)


,fecha_hora,estacion,mp25_ug_m3,mp25_ug_m3_estado
86638,2023-11-18 22:00:00,Ventanas,192.0,no_validado
96160,2024-12-19 16:00:00,Ventanas,189.0,no_validado
96161,2024-12-19 17:00:00,Ventanas,174.0,no_validado
13604,2024-07-20 20:00:00,Centro Quintero,157.0,no_validado
86639,2023-11-18 23:00:00,Ventanas,146.0,no_validado


MP10 (µg/m³N)


,fecha_hora,estacion,mp10_ug_m3n,mp10_ug_m3n_estado
60783,2023-12-07 15:00:00,La Greda,554.0,no_validado
52639,2023-01-02 07:00:00,La Greda,510.0,no_validado
64362,2024-05-04 18:00:00,La Greda,461.0,no_validado
64521,2024-05-11 09:00:00,La Greda,441.0,no_validado
54059,2023-03-02 11:00:00,La Greda,412.0,no_validado


SO₂ (ppb)


,fecha_hora,estacion,so2_ppb,so2_ppb_estado
1777,2023-03-16 01:00:00,Centro Quintero,222.05,no_validado
47793,2025-06-14 09:00:00,Concón,191.94,no_validado
1778,2023-03-16 02:00:00,Centro Quintero,178.35,no_validado
47792,2025-06-14 08:00:00,Concón,138.06,no_validado
1783,2023-03-16 07:00:00,Centro Quintero,107.01,no_validado


## 9. Hallazgos y continuación

Los datos muestran diferencias entre estaciones que dependen del contaminante, perfiles horarios distintos y asociaciones con viento que cambian de signo. Se refinó el foco del proyecto hacia comparaciones por contaminante, estación y horario con cobertura explícita.

El conteo de episodios críticos queda pendiente de verificar definición, unidades, periodos de agregación y validación. También debe confirmarse la convención temporal antes de cruzar nuevas fuentes. La comparación describe estaciones, no toda la población comunal.

La celda siguiente exporta tablas, siete gráficos HTML interactivos, una figura PNG y el informe de hallazgos a las carpetas del proyecto. El script puede ejecutarse por separado con `python src/analizar.py`.

In [11]:
import analizar
resultados = analizar.run(df)
print(resultados)
display(Markdown((ROOT / 'reports/hallazgos.md').read_text(encoding='utf8')))

{'horas_comunes_mp25': 22671, 'graficos_html': 7, 'archivo_hallazgos': 'C:\\Users\\fksot\\Documents\\Codex\\2026-10-04\\an\\outputs\\dataset_cqp\\reports\\hallazgos.md'}


# Hallazgos preliminares del EDA · Aire CQP

Fuente: descarga SINCA del 4 de octubre de 2026. Periodo 2023–2025. Cuatro estaciones históricas. Todas las mediciones disponibles de contaminantes en esta versión están clasificadas como no validadas. Los hallazgos son descriptivos y provisionales.

## Pregunta ajustada

¿Cómo varían las concentraciones horarias de MP2.5, MP10 y SO₂ entre Ventanas, La Greda, Centro Quintero y Concón, según hora del día, época del año y condiciones meteorológicas disponibles, durante 2023–2025?

La frecuencia de episodios críticos queda pendiente de una definición analítica y normativa verificada. Concón sustituye explícitamente el foco Concón Oeste y no se presume identidad entre ambas. El análisis no evalúa un efecto de la nueva red pública.

## Calidad y cobertura

105.216 posiciones hora/estación y 309.217 mediciones de contaminantes disponibles. No se imputan faltantes, no se eliminan extremos dentro de rango y no hay claves duplicadas. La menor cobertura mensual de MP2.5 corresponde a Ventanas, 2025-11: **72.36%** (521/720 horas). Esta pérdida de información puede afectar la comparación y la evolución anual.

## Hallazgos que responden a la pregunta

1. En horas comunes, Ventanas tiene la mayor mediana de MP2.5: **16 µg/m³**. Centro Quintero y La Greda tienen 13 y 13 µg/m³, y Concón 12 µg/m³. Esto describe las estaciones seleccionadas, no una exposición total por comuna.

2. El orden entre estaciones cambia según contaminante. Ventanas presenta la mayor mediana de MP10 y Concón la mayor mediana de SO₂. Por tanto, la pregunta debe tratar cada contaminante por separado.

3. En Ventanas, el perfil mediano de MP2.5 alcanza **24 µg/m³ a las 14:00**, frente a 10 µg/m³ a las 07:00. Los perfiles usan las mismas horas entre estaciones. La diferencia horaria merece explorarse por año y época antes de considerarla estable.

4. La asociación de MP2.5 con velocidad del viento cambia de signo: Spearman **0.235** en Ventanas, **0.096** en Concón, **-0.209** en La Greda y **-0.214** en Centro Quintero. Se calcula con todos los pares disponibles dentro de cada estación. No permite afirmar que el viento cause el cambio ni identificar una fuente emisora.

## MP2.5 (µg/m³)

Comparación sobre **22,671 horas comunes** entre las cuatro estaciones. No se imputan valores.

| estacion | n | media | mediana | p95 | maximo |
| --- | --- | --- | --- | --- | --- |
| Centro Quintero | 22671 | 15.895 | 13.000 | 41.000 | 157.000 |
| Concón | 22671 | 13.855 | 12.000 | 29.000 | 126.000 |
| La Greda | 22671 | 15.874 | 13.000 | 37.000 | 118.000 |
| Ventanas | 22671 | 18.221 | 16.000 | 41.000 | 192.000 |

La mediana describe el centro de la distribución; el percentil 95 describe su cola superior. Los máximos son registros puntuales que requieren revisión, no episodios oficialmente constatados.

## MP10 (µg/m³N)

Comparación sobre **25,223 horas comunes** entre las cuatro estaciones. No se imputan valores.

| estacion | n | media | mediana | p95 | maximo |
| --- | --- | --- | --- | --- | --- |
| Centro Quintero | 25223 | 38.050 | 35.000 | 71.000 | 223.000 |
| Concón | 25223 | 31.871 | 29.000 | 67.000 | 261.000 |
| La Greda | 25223 | 39.024 | 34.000 | 86.000 | 554.000 |
| Ventanas | 25223 | 43.603 | 40.000 | 87.000 | 396.000 |

La mediana describe el centro de la distribución; el percentil 95 describe su cola superior. Los máximos son registros puntuales que requieren revisión, no episodios oficialmente constatados.

## SO₂ (ppb)

Comparación sobre **24,993 horas comunes** entre las cuatro estaciones. No se imputan valores.

| estacion | n | media | mediana | p95 | maximo |
| --- | --- | --- | --- | --- | --- |
| Centro Quintero | 24993 | 3.566 | 2.920 | 7.660 | 222.050 |
| Concón | 24993 | 4.544 | 3.620 | 10.694 | 191.940 |
| La Greda | 24993 | 2.914 | 2.770 | 4.830 | 90.040 |
| Ventanas | 24993 | 2.913 | 2.710 | 4.250 | 87.580 |

La mediana describe el centro de la distribución; el percentil 95 describe su cola superior. Los máximos son registros puntuales que requieren revisión, no episodios oficialmente constatados.

## Meteorología y dimensiones adicionales

| estacion | predictor | n_pares | spearman |
| --- | --- | --- | --- |
| Concón | temperatura_c | 25309 | 0.019 |
| Concón | humedad_relativa_pct | 25309 | -0.102 |

Temperatura y humedad se analizan solo en Concón. La tabla de sectores del viento conserva n por sector y una convención de ocho sectores de 45 grados sobre la dirección reportada en Deg.M. No se calcula una media aritmética de direcciones.

## Límites de interpretación

- Registros de contaminantes no validados: revisar calibraciones y contexto antes de interpretar extremos.
- Las unidades no se convierten ni se mezclan entre contaminantes.
- Las horas conservan la etiqueta original de SINCA, sin offset UTC confirmado.
- Las horas comunes reducen diferencias de disponibilidad, pero el subconjunto puede seguir teniendo sesgo de selección.
- Las observaciones horarias están autocorrelacionadas. Las correlaciones son descriptivas, sin pruebas de significancia ni causalidad.
- Las medias diarias usan al menos 18 horas como criterio de EDA, no normativo.
- Las concentraciones se observan en puntos de monitoreo, no sobre toda la población comunal.

## Siguiente revisión analítica

Comprobar perfiles por año y época del año, investigar los meses con menor cobertura, revisar extremos contra los TXT y evaluar relaciones con dirección del viento. Mantener pendiente la clasificación de episodios hasta verificar el indicador y la validación necesaria.

## Fuentes

[SINCA, estaciones de Valparaíso](https://sinca.mma.gob.cl/index.php/region/index/id/V). [Clasificación de validación](https://sinca.mma.gob.cl/index.php/pagina/index/id/validacion). Las URLs exactas y hashes de cada serie están en docs/download_manifest.json.